# 03 The break I hit in my own loads

The article says I stack monthly publications, which is why this ONS warning stopped me. This notebook
backs that sentence with a number from my own pipeline, recomputed from its files rather than copied from its README.

The break: NHS England changed the cancer waiting times standards from October 2023 data. Its Cancer Waiting
Times page says statistics before October 2023 for the 31-day and 62-day standards "are not directly comparable"
with those after. My Cancer Waiting Times pipeline (`~/Documents/cancer_waiting_time`) loads the monthly combined
files from April 2022 and keeps only October 2023 onwards.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from common import (PAGES, PAGE_SOURCES, POPULATION, PROCESSED, RAW, RETRIEVED, SOURCES, TABLES,
                    page_text, population_16_plus, read_sheet, row, sha256)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
PROCESSED.mkdir(parents=True, exist_ok=True)

In [2]:
import subprocess

CWT = Path.home() / "Documents" / "cancer_waiting_time"
bronze_files = sorted((CWT / "data" / "bronze").glob("cwt_bronze_*.parquet"))
assert bronze_files, f"no Bronze files under {CWT}; this notebook needs the cancer pipeline checked out there"

try:
    cwt_commit = subprocess.run(["git", "-C", str(CWT), "rev-parse", "HEAD"], capture_output=True, text=True,
                                check=True).stdout.strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    cwt_commit = "unknown"

# I apply exactly the two filters my cancer pipeline's Silver notebook applies, in the same order.
df = pd.concat([pd.read_parquet(f) for f in bronze_files], ignore_index=True)
provider = df[df["Basis"] == "Provider"].copy()
provider["period_date"] = pd.to_datetime(provider["Period"], format="mixed", dayfirst=True)
before = provider[provider["period_date"] < "2023-10-01"]
after = provider[provider["period_date"] >= "2023-10-01"]

months = lambda d: d["period_date"].dt.to_period("M")
facts = pd.DataFrame([
    dict(fact="raw rows loaded (all bases)", value=len(df)),
    dict(fact="provider-basis rows", value=len(provider)),
    dict(fact="provider rows before October 2023 (set aside)", value=len(before)),
    dict(fact="share of provider rows set aside (%)", value=round(100 * len(before) / len(provider), 1)),
    dict(fact="months set aside", value=months(before).nunique()),
    dict(fact="first month set aside", value=str(months(before).min())),
    dict(fact="last month set aside", value=str(months(before).max())),
    dict(fact="provider rows kept, October 2023 onwards", value=len(after)),
    dict(fact="months kept", value=months(after).nunique()),
]).assign(source=f"cancer_waiting_time Bronze, commit {cwt_commit[:7]}")
facts

,fact,value,source
0,raw rows loaded (all bases),1076865,"cancer_waiting_time Bronze, commit 7800040"
1,provider-basis rows,765412,"cancer_waiting_time Bronze, commit 7800040"
2,provider rows before October 2023 (set aside),305174,"cancer_waiting_time Bronze, commit 7800040"
3,share of provider rows set aside (%),39.9,"cancer_waiting_time Bronze, commit 7800040"
4,months set aside,18,"cancer_waiting_time Bronze, commit 7800040"
5,first month set aside,2022-04,"cancer_waiting_time Bronze, commit 7800040"
6,last month set aside,2023-09,"cancer_waiting_time Bronze, commit 7800040"
7,"provider rows kept, October 2023 onwards",460238,"cancer_waiting_time Bronze, commit 7800040"
8,months kept,24,"cancer_waiting_time Bronze, commit 7800040"


## Checks

The counts have to match my cancer pipeline's own audit trail (1,076,865 rows loaded, 460,238 kept). The
publisher's statement has to be in the committed snapshot of the NHS England page.

One more thing I check, because it matters for the article: the rows before and after the break use the
**same** standard names. Nothing in the file itself marks the break. The warning is only on the web page,
which is why I snapshot pages.

In [3]:
assert len(df) == 1_076_865, len(df)
assert len(after) == 460_238, len(after)
assert len(provider) == len(before) + len(after)

nhse = page_text("nhse_cancer_waiting_times_landing.html")
statement = ("statistics from before October 2023 for 31 day and 62 day are not directly comparable to those "
             "after October 2023 due to the introduction of new standards")
assert statement in nhse, "NHS England's comparability statement is not in the snapshot"

same_labels = set(before["Standard_or_Item"]) == set(after["Standard_or_Item"])
print("Standard names identical before and after the break:", same_labels)
print(sorted(after["Standard_or_Item"].unique()))

facts = pd.concat([facts, pd.DataFrame([
    dict(fact="standard names identical either side of the break", value=same_labels,
         source="cancer_waiting_time Bronze"),
    dict(fact="NHS England comparability statement", value=statement,
         source="nhse_cancer_waiting_times_landing.html, retrieved " + RETRIEVED),
])], ignore_index=True)
facts.astype({"value": str}).to_csv(PROCESSED / "own_break.csv", index=False)
facts

Standard names identical before and after the break: True
['31D', '62D', 'FDS', 'Urgent Breast Symptomatic referral', 'Urgent Suspected Cancer referral']


,fact,value,source
0,raw rows loaded (all bases),1076865,"cancer_waiting_time Bronze, commit 7800040"
1,provider-basis rows,765412,"cancer_waiting_time Bronze, commit 7800040"
2,provider rows before October 2023 (set aside),305174,"cancer_waiting_time Bronze, commit 7800040"
3,share of provider rows set aside (%),39.9,"cancer_waiting_time Bronze, commit 7800040"
4,months set aside,18,"cancer_waiting_time Bronze, commit 7800040"
5,first month set aside,2022-04,"cancer_waiting_time Bronze, commit 7800040"
6,last month set aside,2023-09,"cancer_waiting_time Bronze, commit 7800040"
7,"provider rows kept, October 2023 onwards",460238,"cancer_waiting_time Bronze, commit 7800040"
8,months kept,24,"cancer_waiting_time Bronze, commit 7800040"
9,standard names identical either side of the break,True,cancer_waiting_time Bronze
